# capit — train on Colab (Stage 3.3)

A thin launcher: all the logic lives in `train.py` in the repo. This notebook clones it, stages the data, and runs the CLI on the T4.

**Before you run:**
1. **Runtime → Change runtime type → T4 GPU.**
2. Locally: `python pipeline/scripts/make_train_zip.py` → builds `data/flickr8k_colab.zip` (Images + dataset_flickr8k.json + vocab.json).
3. **Push your latest code** — the notebook runs what is in git, not your local working tree.

Get the zip onto Drive **once** (either the upload cell below, or drag it into `MyDrive/capit/` via the Drive website — more reliable for ~1 GB). After that, reconnects just re-stage from Drive and `--resume auto` continues the run.

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
# torch/torchvision are preinstalled on Colab; this pulls the small extras (nltk, ...).
# No -q: a failed install must be visible, not surface 3 hours later as ModuleNotFoundError.
!rm -rf /content/capit && git clone https://github.com/Bukunmi2108/capit.git /content/capit
!pip install -e /content/capit/pipeline
import capit  # fail fast if the install didn't take
print("capit installed")

Cloning into '/content/capit'...
remote: Enumerating objects: 149, done.
remote: Counting objects: 100% (149/149), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 149 (delta 55), reused 132 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (149/149), 96.33 KiB | 6.02 MiB/s, done.
Resolving deltas: 100% (55/55), done.
Obtaining file:///content/capit/pipeline
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for capit (pyproject.toml) ... done
  Created wheel for capit: filename=capit-0.1.0-0.editable-py3-none-any.whl size=1307 sha256=fbe47f272d2e771f315dc44349cd36179ae23791926a0b0626981391c9c3976b
  Stored in directory: /tmp/pip-ephem-wheel-cache-630q9_cb/wheels/ff/0a/93/9457afa2ecf0d24429d3eb1754c723aeaf950fdbae03614da9
Successfully built capit
  Attempting uninstall: capit
    Fou

capit installed


In [7]:
import os, shutil
dest = '/content/drive/MyDrive/capit/flickr8k_colab.zip'
if os.path.exists(dest):
    print('already on Drive — skipping upload')
else:
    from google.colab import files
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    up = files.upload()  # pick flickr8k_colab.zip
    shutil.move(next(iter(up)), dest)
    print('uploaded and stashed on Drive:', dest)

already on Drive — skipping upload


In [8]:
# Copy OFF the Drive mount to local disk, then unzip (never read images over Drive — slow).
# `&&` so unzip only runs if the copy succeeded.
!cp /content/drive/MyDrive/capit/flickr8k_colab.zip /content/flickr8k_colab.zip && unzip -q -o /content/flickr8k_colab.zip -d /content/flickr8k
import os
n = len(os.listdir('/content/flickr8k/Images'))
assert n == 8091, f"expected 8091 images, got {n} — bad/partial zip"
print(n, "images staged")

8091 images staged


In [9]:
!python -m capit.train \
  --data-root /content/flickr8k \
  --vocab-path /content/flickr8k/vocab.json \
  --ckpt-dir /content/drive/MyDrive/capit/checkpoints \
  --resume auto

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100% 97.8M/97.8M [00:00<00:00, 147MB/s] 
epoch 0  loss 4.8991  val_bleu4 8.18  best 8.18 *  | a boy in a blue shirt and a white shirt is standing on a skateboard
epoch 1  loss 4.0486  val_bleu4 17.58  best 17.58 *  | a boy is sitting on a bench
epoch 2  loss 3.7289  val_bleu4 17.30  best 17.58  | a man in a red shirt and black pants is sitting on a bench
epoch 3  loss 3.5054  val_bleu4 17.85  best 17.85 *  | a man in a red shirt is sitting on a skateboard
epoch 4  loss 3.3269  val_bleu4 19.32  best 19.32 *  | a man in a red shirt is sitting on a skateboard
epoch 5  loss 3.1763  val_bleu4 18.90  best 19.32  | a man in a red shirt is sitting on a skateboard
epoch 6  loss 3.0429  val_bleu4 18.59  best 19.32  | a man in a red shirt is riding on a skateboard
epoch 7  loss 2.9214  val_bleu4 19.62  best 19.62 *  | a man in a red shirt is riding on a swing
epoch 

## If disconnected
Reconnect → re-run **all** cells. The upload cell skips (zip already on Drive), the data cell re-stages, and `--resume auto` loads `latest.pt` from Drive and continues from the next epoch — checkpoints live on Drive, so a disconnect costs minutes, not the run. (train.py refuses to run if `--ckpt-dir` points at an unmounted Drive.)

**Exit gate (Stage 3.3):** `best.pt` on Drive with val BLEU-4 (greedy, nltk) ≥ ~14. Download `MyDrive/capit/checkpoints/best.pt` for Phase 4.